# 02 · Feature Engineering

Loads the raw match data and produces the model-ready feature set.

**Key transforms**
- Rolling form windows (10, 20, 38 matches) for points, shots, goals, discipline, corners
- Head-to-head points from last 2 encounters
- Pre-match league rank
- Differential features (home − away) for each rolling stat

All windows use `shift(1)` before aggregation to ensure no future information
leaks into any row.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from src.features.engineering import select_raw_columns, build_features, FEATURE_COLUMNS

RAW_PATH       = pathlib.Path("../data/raw/EPL_LaLiga_Data.csv")
PROCESSED_PATH = pathlib.Path("../data/processed/matches_features.csv")

## Load raw data

In [2]:
raw = pd.read_csv(RAW_PATH, parse_dates=["Date"])
raw = select_raw_columns(raw)
print(raw.shape)
raw.head(3)

(15201, 23)


/var/folders/5d/1xw4nstd6ls9kr2sdhvp91l40000gn/T/ipykernel_94600/2962673938.py:1: DtypeWarning: Columns (78) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(RAW_PATH, parse_dates=["Date"])


,Div,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
0,E0,2005/2006,2005-08-13,Aston Villa,Bolton,2.0,2.0,D,2.0,2.0,...,2.0,6.0,14.0,16.0,7.0,8.0,0.0,2.0,0.0,0.0
1,E0,2005/2006,2005-08-13,Everton,Man United,0.0,2.0,A,0.0,1.0,...,5.0,5.0,15.0,14.0,8.0,6.0,3.0,1.0,0.0,0.0
2,E0,2005/2006,2005-08-13,Fulham,Birmingham,0.0,0.0,D,0.0,0.0,...,7.0,4.0,12.0,13.0,6.0,6.0,1.0,2.0,0.0,0.0


## Run feature engineering pipeline

In [3]:
features = build_features(raw, drop_na=True)
print(f"Rows after dropping NaN early-season matches: {len(features):,}")
features.shape

Rows after dropping NaN early-season matches: 15,072


(15072, 59)

## Inspect features

In [4]:
features[FEATURE_COLUMNS].describe().round(2)

,RollingPointsDif10,HomeTeamPointsLast2Encounters,HomeRollingPoints38,AwayRollingPoints38,RollingPointsDiff38,HomeRollingShotsTarget20,AwayRollingShotsTarget20,HomeRollingShots20,AwayRollingShots20,HomeRollingYellowCards20,...,AwayRollingReceivedGoals20,RollingShotsDif20,RollingShotsTargetDif20,RollingYellowCardsDif20,RollingRedCardsDif20,RollingFoulsDif20,RollingCornersDif20,HomeRank,AwayRank,RankDif
count,15072.00,15072.00,15072.00,15072.00,15072.00,15072.00,15072.00,15072.00,15072.00,15072.00,...,15072.00,15072.00,15072.00,15072.00,15072.00,15072.00,15072.00,15072.00,15072.00,15072.00
mean,5.00,2.28,57.15,39.31,17.84,102.39,80.46,261.61,208.17,37.15,...,28.85,53.44,21.93,-5.24,-0.49,-2.66,24.44,5.28,5.18,0.09
std,8.07,2.01,22.47,19.24,27.45,37.81,29.21,70.89,57.35,13.89,...,9.51,79.11,39.06,13.73,2.18,65.65,33.66,3.85,3.83,4.52
min,-26.00,0.00,0.00,0.00,-88.00,0.00,0.00,3.00,1.00,0.00,...,0.00,-355.00,-164.00,-74.00,-11.00,-430.00,-142.00,1.00,1.00,-19.00
25%,0.00,0.00,46.00,27.00,0.00,80.00,64.00,233.00,185.00,27.00,...,23.00,11.00,0.00,-14.00,-2.00,-33.00,5.00,2.00,2.00,-3.00
50%,5.00,2.00,57.00,37.00,18.00,96.00,76.00,262.00,209.00,36.00,...,29.00,52.00,20.00,-5.00,0.00,-3.00,24.00,4.00,4.00,0.00
75%,11.00,4.00,71.00,50.00,37.00,122.00,95.00,301.00,238.00,47.25,...,35.00,97.00,43.00,3.00,1.00,27.00,45.00,8.00,7.00,3.00
max,30.00,6.00,110.00,94.00,101.00,254.00,200.00,497.00,399.00,83.00,...,63.00,391.00,232.00,63.00,9.00,395.00,178.00,20.00,20.00,19.00


In [5]:
# Check for remaining NaNs in feature columns
features[FEATURE_COLUMNS].isna().sum().to_frame("null_count").query("null_count > 0")

,null_count


In [6]:
# Outcome distribution
features["FTR"].value_counts(normalize=True).round(3)

FTR
H    0.462
A    0.293
D    0.246
Name: proportion, dtype: float64

## Save processed dataset

In [7]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(PROCESSED_PATH, index=False)
print(f"Saved {len(features):,} rows → {PROCESSED_PATH}")

Saved 15,072 rows → ../data/processed/matches_features.csv
